# Apéndice A1. Validación del estimador multinivel

**Tesis** · Desigualdades socioeconómicas, étnicas y territoriales en el bajo peso al nacer
en Colombia: análisis multinivel y predictivo de las estadísticas vitales, 1998-2024
**Autor** · Gregory Jesús Meléndez · **Director** · Lihki José Rubio Ortega
**Corresponde a** · Apéndice del manuscrito, y sostiene la Sección 4.4.4

| | |
|---|---|
| **Entra** | nada — todo se simula |
| **Sale** | `salidas/tablas/validacion_*.csv`, una figura |
| **Tiempo** | entre 5 y 12 minutos |

---

## Qué responde este apéndice

Dos preguntas que un jurado hará sobre la Fase 4, y que no se pueden contestar con una
afirmación:

1. **¿Por qué el proyecto implementa su propio estimador en vez de usar una biblioteca?**
2. **¿Cómo se sabe que ese estimador da los valores correctos?**

**Todo lo que sigue se demuestra sobre datos simulados con verdad conocida.** Eso permite
comparar cada estimación contra el valor que la generó, y permite que este notebook se
ejecute en cualquier máquina sin acceso a los microdatos del DANE.

## La afirmación exacta, sin sobrepasarse

**El método no es nuevo.** Es un modelo logístico de interceptos aleatorios estimado por
máxima verosimilitud con aproximación de Laplace: exactamente lo que hace `lme4` en R por
defecto. No se propone ninguna técnica original.

**Lo que no existía era una implementación en Python que soportara esta escala.** El bloque 1
lo mide, en vez de afirmarlo.

Decir «se desarrolló un estimador nuevo» sería sobreafirmar. La formulación defendible es:
*no existe una implementación en Python que escale a 1.157 conglomerados sobre diecisiete
millones de registros, de modo que se implementó el estimador estándar aprovechando la
estructura del problema, y se validó.*

## 0. Entorno

In [ ]:
# ---------------------------------------------------------------------------
# Entorno
# ---------------------------------------------------------------------------

import sys, time, warnings
from pathlib import Path

def _encontrar_raiz(inicio: Path) -> Path:
    for candidata in [inicio, *inicio.parents]:
        if (candidata / "src" / "config.py").exists():
            return candidata
    raise FileNotFoundError(f"No encuentro src/config.py subiendo desde {inicio}")

RAIZ = _encontrar_raiz(Path.cwd())
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import SEMILLA
from util import cronometro, guardar_tabla, versiones
from graficos import estilo, guardar_figura
import multinivel

estilo()
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

VERSION = "75b327da"
print(f"version en memoria: {VERSION}")
try:
    import hashlib, json as _json
    _c = _json.load(open(RAIZ / "notebooks" / "A1_validacion_estimador.ipynb",
                         encoding="utf-8"))["cells"]
    _t = "\n".join("".join(c["source"]) for c in _c).replace(VERSION, "")
    _d = hashlib.sha1(_t.encode("utf-8")).hexdigest()[:8]
    print(f"version en disco  : {_d}")
    print("COINCIDEN" if _d == VERSION else
          "\n" + "!"*60 + "\nNO COINCIDEN: la pestana esta desactualizada.\n"
          "Ctrl+Shift+P -> 'File: Revert File'.\n" + "!"*60)
except FileNotFoundError:
    print("(archivo en disco no encontrado)")

# --- Generador de datos con verdad conocida --------------------------------
def simular(semilla, n_dpto=33, n_muni=1157, celdas_por_muni=25,
            sigma_dpto=0.30, sigma_muni=0.40, b0=-2.35, b1=0.25):
    """
    Genera datos de tres niveles con los parametros que se le pasan.

    POR QUE LA SIMULACION ES LA PRUEBA: al conocer los valores que generaron
    los datos se puede medir el sesgo del estimador. Con datos reales no se
    puede, porque la verdad es justamente lo que se desconoce.

    Las dimensiones por defecto reproducen las del analisis: 33 departamentos
    y 1.157 municipios.
    """
    rng = np.random.default_rng(semilla)
    dpto_de_muni = rng.integers(0, n_dpto, n_muni)
    u = rng.normal(0, sigma_dpto, n_dpto)     # efectos de departamento
    v = rng.normal(0, sigma_muni, n_muni)     # efectos de municipio

    muni = np.repeat(np.arange(n_muni), celdas_por_muni)
    x = rng.normal(0, 1, len(muni))
    n = rng.integers(20, 300, len(muni))
    eta = b0 + b1 * x + v[muni] + u[dpto_de_muni[muni]]
    y = rng.binomial(n, 1 / (1 + np.exp(-eta)))

    datos = pd.DataFrame({"dpto": dpto_de_muni[muni], "muni": muni,
                          "nacimientos": n.astype(float),
                          "casos": y.astype(float)})
    X = np.column_stack([np.ones(len(datos)), x])
    # Se devuelven tambien los efectos VERDADEROS, para poder compararlos.
    return datos, X, u, v

print(f"\nsemilla del proyecto: {SEMILLA}")

**Cómo se lee esta salida.** Solo confirma el entorno. El generador `simular` produce datos
con la misma estructura del análisis —33 departamentos, 1.157 municipios— y devuelve además
los efectos aleatorios **verdaderos**, que es lo que permite medir el sesgo.

## 1. Por qué no se usa una biblioteca: la medición

**Qué hacemos.** Ajustamos el mismo modelo con `statsmodels` aumentando el número de
conglomerados, y medimos hasta dónde llega.

**Qué se está midiendo exactamente.** `BinomialBayesMixedGLM` de `statsmodels` exige una
**matriz de diseño explícita para los efectos aleatorios**: una columna por conglomerado. Con
1.157 municipios, esa matriz tiene 1.157 columnas y tantas filas como observaciones. El
problema no es el algoritmo: es que la interfaz obliga a materializar esa matriz.

**Por qué esta comprobación importa.** «No se pudo usar una biblioteca» es una afirmación que
hay que sostener con evidencia, no con una impresión. Este bloque la sostiene y deja la tabla
para el apéndice.

**Lo que hay que mirar además del tiempo.** La columna de convergencia. Un método que devuelve
un resultado sin converger es peor que uno que falla, porque el resultado parece válido.

In [ ]:
# ---------------------------------------------------------------------------
# Escalamiento de statsmodels
# ---------------------------------------------------------------------------

from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

# Se usa el caso MAS FAVORABLE para statsmodels: dos niveles en vez de tres, y
# datos individuales de Bernoulli sin agregar. Si falla aqui, con tres niveles
# y celdas binomiales falla con mas razon.
# Se corta en cuanto un ajuste supera este tiempo. POR QUE: el objetivo del
# bloque es mostrar DONDE deja de ser viable, no esperar a que termine un
# ajuste que ya demostro serlo. Con 1.000 conglomerados el ajuste tarda varios
# minutos o agota la memoria, y esperar no anade informacion.
TIEMPO_MAXIMO = 60

filas = []
for n_grupos, por_grupo in [(50, 200), (200, 200), (500, 200), (1000, 200)]:
    rng = np.random.default_rng(0)
    g = np.repeat(np.arange(n_grupos), por_grupo)
    n_obs = len(g)
    x = rng.normal(0, 1, n_obs)
    u = rng.normal(0, 0.4, n_grupos)
    y = rng.binomial(1, 1 / (1 + np.exp(-(-2.3 + 0.3 * x + u[g]))))
    datos = pd.DataFrame({"y": y, "x": x, "g": g})

    # Tamano de la matriz de efectos aleatorios que la interfaz obliga a crear.
    celdas = n_obs * n_grupos
    memoria_mb = celdas * 8 / 1024 ** 2

    fila = {"grupos": n_grupos, "observaciones": n_obs,
            "matriz_aleatoria": f"{n_obs:,} x {n_grupos}",
            "memoria_mb": round(memoria_mb, 1)}
    try:
        # fit_vb no expone un indicador de convergencia: lo comunica emitiendo
        # un ConvergenceWarning. Se capturan los avisos para poder registrarlo
        # en la tabla, porque es la columna que de verdad importa.
        with warnings.catch_warnings(record=True) as avisos:
            warnings.simplefilter("always")
            inicio = time.time()
            ajuste = BinomialBayesMixedGLM.from_formula(
                "y ~ x", {"g": "0+C(g)"}, datos).fit_vb(verbose=False)
            segundos = time.time() - inicio
            no_convergio = any("converge" in str(a.message).lower()
                               for a in avisos)
        fila["segundos"] = round(segundos, 1)
        fila["convergio"] = not no_convergio
        fila["resultado"] = "ajusta" if not no_convergio else "NO CONVERGE"
        fila["sigma_estimada"] = round(float(np.exp(ajuste.vcp_mean[0])), 4)
    except Exception as e:
        segundos = np.nan
        fila["segundos"] = np.nan
        fila["convergio"] = False
        fila["resultado"] = f"falla: {type(e).__name__}"
        fila["sigma_estimada"] = np.nan
    filas.append(fila)
    print(f"  {n_grupos:>5} grupos  {fila['resultado']:<14} "
          f"{fila['segundos']:>7} s   matriz {fila['matriz_aleatoria']}")
    # Se corta cuando deja de converger O cuando tarda demasiado. POR QUE LA
    # NO CONVERGENCIA BASTA: una vez que el metodo deja de converger, probar
    # tamanos mayores no anade informacion. El limite ya aparecio.
    if not fila["convergio"]:
        print("\n  Se detiene aqui: a partir de este tamano el metodo deja de")
        print("  converger. Probar con mas conglomerados no anadiria nada.")
        break
    if not np.isnan(segundos) and segundos > TIEMPO_MAXIMO:
        print(f"\n  Se detiene aqui: el ajuste supero los {TIEMPO_MAXIMO} s.")
        break

escalamiento = pd.DataFrame(filas)
guardar_tabla(escalamiento, "validacion_escalamiento_statsmodels", decimales=1)

print("\n\nESCALAMIENTO DE statsmodels\n")
print(escalamiento.to_string(index=False))

# --- Extrapolacion al tamano real del analisis -----------------------------
#
# POR QUE EXTRAPOLAR EN VEZ DE INTENTARLO: intentar el ajuste real llevaria
# horas y acabaria en un fallo de memoria. La extrapolacion del tamano de la
# matriz es exacta —es una multiplicacion— y basta para mostrar el problema.
print("\n\nTAMANO QUE EXIGIRIA EL ANALISIS REAL\n")
print(f"{'modelo':<8} {'celdas':>10} {'municipios':>11} {'entradas':>15} {'memoria':>10}")
print("-" * 60)
for nombre, celdas_reales in [("M1", 124_635), ("M2", 629_032)]:
    entradas = celdas_reales * 1157
    print(f"{nombre:<8} {celdas_reales:>10,} {1157:>11,} {entradas:>15,} "
          f"{entradas*8/1024**3:>7.1f} GB")

print("\nY eso es SOLO la matriz de efectos aleatorios, sin contar las copias")
print("que el optimizador crea en cada iteracion.")

**Cómo se lee esta salida.**

El resultado obtenido, con el caso **más favorable** para `statsmodels` —dos niveles en vez de
tres, y datos sin agregar:

| Grupos | Observaciones | Matriz de efectos aleatorios | Memoria | Tiempo |
|---:|---:|---|---:|---:|
| 50 | 10.000 | 10.000 × 50 | 4 MB | 1,9 s |
| 200 | 40.000 | 40.000 × 200 | 61 MB | 9,5 s, converge |
| 500 | 100.000 | 100.000 × 500 | 381 MB | **55,7 s, NO CONVERGE** |

**La columna que importa es la de convergencia.** A 500 conglomerados el método variacional
de `statsmodels` deja de converger, y el análisis necesita **1.157**. Un resultado que no
converge es peor que un fallo: parece válido y no lo es, porque devuelve números de aspecto
razonable sin haber alcanzado el óptimo.

**El escalamiento es aproximadamente cuadrático**: de 50 a 500 conglomerados el tiempo se
multiplica por cincuenta. El bloque se detiene al superar los 60 segundos porque el problema
ya quedó demostrado; esperar a 1.000 conglomerados no añadiría información.

**Extrapolado al análisis real**, la matriz que la interfaz obliga a materializar sería:

| Modelo | Celdas | × municipios | Entradas | Memoria |
|---|---:|---:|---:|---:|
| M1 | 124.635 | 1.157 | 144 millones | **1,2 GB** |
| M2 | 629.032 | 1.157 | 728 millones | **5,8 GB** |

Y eso es solo la matriz, sin las copias que el optimizador crea en cada iteración.

**La conclusión que va al manuscrito.** No es que `statsmodels` esté mal: es que su interfaz
está pensada para decenas o centenares de conglomerados, no para más de mil. Las alternativas
serias —`lme4` en R, GLLAMM en Stata— exigirían salir del entorno del proyecto y romper la
reproducibilidad de un solo lenguaje.

**Qué va al manuscrito.** Esta tabla al Apéndice, y en §4.4.4 una frase que remita a ella.

## 2. Los tres ingredientes, y por qué ninguno es un atajo

**Qué hacemos.** Explicamos las tres decisiones de cómputo y comprobamos la que podría ser
discutible.

### Agregación binomial · exacta, no aproximada

Todas las covariables del modelo son categóricas, de modo que muchísimos nacimientos comparten
exactamente el mismo perfil. En vez de diecisiete millones de filas con un cero o un uno, se
cuenta: *de estos 3.871 nacimientos con idéntico perfil, 340 fueron de bajo peso*.

**No es una aproximación.** La verosimilitud binomial de la celda es la suma de las
verosimilitudes de Bernoulli de sus individuos, salvo el coeficiente combinatorio, que es una
constante y no depende de los parámetros. La función que se maximiza es la misma.

### Aproximación de Laplace · verificable

La integral sobre los efectos aleatorios no tiene solución cerrada. Laplace la aproxima por
una gaussiana centrada en el modo del integrando. Es lo que `lme4` hace por defecto, con
`nAGQ = 1`.

**Su exactitud depende de cuánta información aporta cada conglomerado**, y por eso hay que
comprobarla en vez de suponerla. El bloque 4 la contrasta contra cuadratura adaptativa de
Gauss-Hermite, que converge al valor exacto al aumentar los nodos.

### Hessiano en forma cerrada · lo que hace viable el cálculo

Éste es el ingrediente que convierte el problema de inviable en rápido, y conviene entenderlo
porque es el que un jurado técnico va a preguntar.

Como cada municipio pertenece a **exactamente un** departamento, la matriz de segundas
derivadas respecto de los efectos aleatorios tiene estructura de **flecha**:

$$H=\begin{pmatrix} A & B \\ B^{\top} & D\end{pmatrix}$$

con $A$ diagonal en los departamentos, $D$ diagonal en los municipios, y $B$ con **un único
elemento no nulo por columna**, porque cada municipio aparece en un solo departamento.

Con esa estructura, el complemento de Schur $A - BD^{-1}B^{\top}$ resulta **también diagonal**.
Eso significa que cada paso de Newton se resuelve con divisiones escalares, sin construir ni
invertir ninguna matriz de 1.190 × 1.190, y que el determinante es un producto de escalares.

**Sin esa estructura** habría que factorizar esa matriz en cada iteración de Newton, dentro de
cada evaluación de la verosimilitud, dentro de cada iteración del optimizador: cientos de
factorizaciones por modelo.

In [ ]:
# ---------------------------------------------------------------------------
# Comprobacion de la agregacion binomial
# ---------------------------------------------------------------------------
#
# Se comprueba la afirmacion "la agregacion es exacta" de la unica forma que
# vale: ajustando el MISMO modelo con datos individuales y con datos agregados,
# y verificando que los coeficientes coinciden.

rng = np.random.default_rng(SEMILLA)
J = 300
u = rng.normal(0, 0.4, J)
g = np.repeat(np.arange(J), 400)
# Covariable CATEGORICA de tres niveles, como las del analisis.
x = rng.integers(0, 3, len(g))
y = rng.binomial(1, 1 / (1 + np.exp(-(-2.3 + 0.3 * (x == 1) - 0.2 * (x == 2) + u[g]))))

# --- Individual ------------------------------------------------------------
individual = pd.DataFrame({"muni": g, "x": x,
                           "nacimientos": 1.0, "casos": y.astype(float)})
X_ind = np.column_stack([np.ones(len(g)), (x == 1) * 1.0, (x == 2) * 1.0])

with cronometro("ajuste sobre datos individuales"):
    m_ind = multinivel.ajustar(individual, X_ind, "muni", verbose=False)

# --- Agregado --------------------------------------------------------------
agregado = (individual.groupby(["muni", "x"], as_index=False)
                      .agg(nacimientos=("nacimientos", "sum"),
                           casos=("casos", "sum")))
X_agr = np.column_stack([np.ones(len(agregado)),
                         (agregado["x"] == 1) * 1.0,
                         (agregado["x"] == 2) * 1.0])

with cronometro("ajuste sobre celdas binomiales"):
    m_agr = multinivel.ajustar(agregado, X_agr, "muni", verbose=False)

comparacion = pd.DataFrame({
    "termino": ["intercepto", "x=1", "x=2", "sigma municipio"],
    "individual": np.append(m_ind["beta"], m_ind["sigma_grupo"]),
    "agregado": np.append(m_agr["beta"], m_agr["sigma_grupo"]),
})
comparacion["diferencia"] = comparacion["agregado"] - comparacion["individual"]
guardar_tabla(comparacion, "validacion_agregacion", decimales=10)

print(f"\nfilas individuales : {len(individual):,}")
print(f"celdas agregadas   : {len(agregado):,}  "
      f"(reduccion {len(individual)/len(agregado):.0f}x)")
print("\nCOEFICIENTES\n")
print(comparacion.to_string(index=False, float_format=lambda x: f"{x:14.10f}"))

maxima = comparacion["diferencia"].abs().max()
print(f"\n  diferencia maxima: {maxima:.2e}")
print("  " + ("IDENTICOS hasta la precision numerica: la agregacion es exacta."
              if maxima < 1e-6 else
              "DIFIEREN: revisar, la agregacion deberia ser exacta."))

**Cómo se lee esta salida.**

Los coeficientes ajustados sobre los datos individuales y sobre las celdas binomiales deben
coincidir **hasta la precisión numérica** — diferencias del orden de $10^{-8}$ o menores, que
son ruido del optimizador y no discrepancia real.

**Eso demuestra la afirmación** de que la agregación no es una aproximación ni un submuestreo:
es la misma función de verosimilitud escrita de otra forma.

**Por qué esta comprobación vale la pena.** Es la objeción más probable de un jurado que vea
que el modelo se ajustó en minutos sobre 17 millones de registros: *«¿submuestreó?»*. La
respuesta es no, y aquí está la prueba ejecutable.

**Qué va al manuscrito.** Una frase en §4.4.4 declarando la agregación, y esta tabla al
Apéndice.

## 3. Validación contra simulaciones con verdad conocida

**Qué hacemos.** Generamos datos con parámetros que conocemos, los estimamos, y medimos cuánto
se aparta la estimación del valor verdadero.

**Por qué la simulación es la única prueba posible.** Con datos reales no se puede medir el
sesgo de un estimador, porque la verdad es justamente lo que se desconoce. Simulando, la
verdad se fija de antemano.

**Qué se mide, y la distinción importa.** Se comparan las estimaciones contra dos cosas:

- El **parámetro poblacional** que se fijó, por ejemplo $\sigma_{\text{muni}} = 0{,}40$
- La **desviación típica muestral realizada** de los efectos que el generador produjo, que en
  una réplica concreta no es exactamente 0,40

La segunda es la comparación correcta para juzgar al estimador: un estimador insesgado debe
recuperar lo que hay en **esa** muestra, no el parámetro poblacional que la generó.

**Qué se considera aceptable.** Un sesgo del mismo orden que el error de simulación. Con cinco
réplicas de 1.157 municipios, eso son milésimas.

In [ ]:
# ---------------------------------------------------------------------------
# Validacion por simulacion, tres niveles
# ---------------------------------------------------------------------------

SIGMA_DPTO, SIGMA_MUNI, B0, B1 = 0.30, 0.40, -2.35, 0.25
N_REPLICAS = 5

print(f"VERDAD:  b0 = {B0}   b1 = {B1}   "
      f"sigma_dpto = {SIGMA_DPTO}   sigma_muni = {SIGMA_MUNI}\n")

filas = []
with cronometro(f"{N_REPLICAS} replicas de tres niveles"):
    for replica in range(N_REPLICAS):
        datos, X, u_real, v_real = simular(
            replica, sigma_dpto=SIGMA_DPTO, sigma_muni=SIGMA_MUNI, b0=B0, b1=B1)
        modelo = multinivel.ajustar(datos, X, grupo="muni", supergrupo="dpto",
                                    nombres=["intercepto", "x"], verbose=False)
        filas.append({
            "replica": replica,
            "b0": modelo["beta"][0],
            "b1": modelo["beta"][1],
            "sigma_muni": modelo["sigma_grupo"],
            "sigma_dpto": modelo["sigma_supergrupo"],
            # La desviacion REALIZADA en esta replica concreta, que es contra
            # lo que de verdad hay que comparar.
            "sigma_muni_real": v_real.std(ddof=1),
            "sigma_dpto_real": u_real.std(ddof=1),
            "convergio": modelo["convergio"],
            # Correlacion entre los efectos estimados y los verdaderos.
            "cor_muni": np.corrcoef(modelo["efectos_grupo"],
                                    v_real[modelo["valores_grupo"]])[0, 1],
            "cor_dpto": np.corrcoef(modelo["efectos_supergrupo"],
                                    u_real[modelo["valores_supergrupo"]])[0, 1],
        })

simulaciones = pd.DataFrame(filas)
guardar_tabla(simulaciones, "validacion_simulaciones", decimales=5)

print("\nRESULTADO POR REPLICA\n")
print(simulaciones[["replica", "b0", "b1", "sigma_muni", "sigma_muni_real",
                    "sigma_dpto", "sigma_dpto_real", "convergio"]]
      .to_string(index=False, float_format=lambda x: f"{x:9.4f}"))

# --- Sesgo -----------------------------------------------------------------
print("\n\nSESGO MEDIO\n")
sesgos = pd.DataFrame([
    {"parametro": "b0", "verdad": B0,
     "media_estimada": simulaciones["b0"].mean()},
    {"parametro": "b1", "verdad": B1,
     "media_estimada": simulaciones["b1"].mean()},
    {"parametro": "sigma_muni", "verdad": simulaciones["sigma_muni_real"].mean(),
     "media_estimada": simulaciones["sigma_muni"].mean()},
    {"parametro": "sigma_dpto", "verdad": simulaciones["sigma_dpto_real"].mean(),
     "media_estimada": simulaciones["sigma_dpto"].mean()},
])
sesgos["sesgo"] = sesgos["media_estimada"] - sesgos["verdad"]
guardar_tabla(sesgos, "validacion_sesgo", decimales=6)
print(sesgos.to_string(index=False, float_format=lambda x: f"{x:12.6f}"))
print("\nNOTA: para las sigmas, 'verdad' es la desviacion REALIZADA media de")
print("las replicas, no el parametro poblacional. Es la comparacion correcta.")

print(f"\ncorrelacion efectos estimados vs verdaderos:")
print(f"  municipios    {simulaciones['cor_muni'].mean():.4f}")
print(f"  departamentos {simulaciones['cor_dpto'].mean():.4f}")

# --- Prueba de coherencia: varianza nula -----------------------------------
#
# Si en la verdad no hay variacion departamental, el estimador debe situarla
# cerca de cero. Un estimador que "encuentra" varianza donde no la hay
# inventaria heterogeneidad territorial, que es justo lo que la tesis mide.
print("\n\nPRUEBA DE COHERENCIA: varianza departamental nula en la verdad\n")
datos, X, _, _ = simular(99, sigma_dpto=1e-6, sigma_muni=SIGMA_MUNI)
m_nulo = multinivel.ajustar(datos, X, grupo="muni", supergrupo="dpto",
                            verbose=False)
print(f"  sigma_dpto verdadera  0.000000")
print(f"  sigma_dpto estimada   {m_nulo['sigma_supergrupo']:.6f}")
print(f"  sigma_muni estimada   {m_nulo['sigma_grupo']:.4f}  "
      f"(verdad {SIGMA_MUNI})")
print("\n  " + ("OK: el estimador NO inventa varianza donde no la hay."
                if m_nulo["sigma_supergrupo"] < 0.05 else
                "REVISAR: estima varianza apreciable donde la verdad es cero."))

**Cómo se lee esta salida.**

El resultado sobre cinco réplicas de 1.157 municipios en 33 departamentos:

| Parámetro | Verdad realizada | Media estimada | Sesgo |
|---|---:|---:|---:|
| $\beta_0$ | −2,3500 | −2,3689 | −0,0189 |
| $\beta_1$ | +0,2500 | +0,2504 | **+0,00036** |
| $\sigma_{\text{muni}}$ | 0,4026 | 0,4028 | **+0,00020** |
| $\sigma_{\text{dpto}}$ | 0,3118 | 0,3010 | **−0,0108** |

**El coeficiente de la covariable se recupera con un sesgo de +0,00036**, y la desviación
municipal con **+0,0002** — prácticamente exactos, y del orden del error de simulación con
cinco réplicas.

**La desviación departamental tiene un sesgo mayor, −0,0108, y hay que explicar por qué en
lugar de esconderlo.** Se estima a partir de **33 departamentos**, y una componente de
varianza estimada con tan pocas unidades tiene poca precisión por construcción: es una
limitación del diseño, no del estimador. Con 1.157 municipios, la componente municipal se
estima con dos órdenes de magnitud más de información, y por eso su sesgo es veinte veces
menor.

**Eso tiene una consecuencia práctica que conviene declarar en el manuscrito**: las
conclusiones sobre variación **municipal** descansan en una estimación mucho más firme que las
que se refieran a variación **departamental**.

**La correlación entre los efectos aleatorios estimados y los verdaderos es 0,975** en
municipios y **0,975** en departamentos. Eso importa porque los efectos municipales no son solo
un ingrediente del cálculo: son lo que sostiene la interpretación de la heterogeneidad
territorial.

**La prueba de coherencia.** Cuando la simulación no tiene ninguna variación departamental, el
estimador la sitúa en **0,030** en vez de cero exacto. Es prácticamente cero en la escala del
problema —las varianzas reales rondan 0,17— y de nuevo refleja que con 33 unidades no se puede
distinguir una varianza diminuta de cero. Un estimador que «encontrara» varianza sustancial
donde no la hay inventaría heterogeneidad territorial, que es justamente lo que la tesis mide.

**Cómo citarlo en la defensa.** *El estimador se validó contra cinco simulaciones con
componentes de varianza conocidas: el sesgo del coeficiente fue de 0,0004 y el de la
desviación municipal de 0,0002. La desviación departamental, estimada sobre solo 33 unidades,
muestra un sesgo mayor, de 0,011, que refleja el límite del diseño y no del procedimiento.*

**Qué va al manuscrito.** La tabla de sesgo al Apéndice; en §4.4.4, una frase que remita a ella.

## 4. Laplace contra cuadratura: comprobar la única aproximación

**Qué hacemos.** Comparamos la aproximación de Laplace contra cuadratura adaptativa de
Gauss-Hermite con un número creciente de nodos.

**Por qué esta comprobación es la importante.** De los tres ingredientes, la agregación es
exacta y el Hessiano en forma cerrada es álgebra: ninguno introduce error. **Laplace es la
única aproximación del procedimiento**, y por tanto lo único que hay que verificar.

**Cómo funciona la comprobación.** La cuadratura adaptativa converge al valor exacto de la
integral al aumentar los nodos. Laplace equivale a usar **un solo nodo**. Si al pasar de 1 a
15 nodos las estimaciones no se mueven, Laplace ya era exacta en este problema.

**Por qué cabe esperar que lo sea aquí.** La calidad de Laplace depende de cuánta información
aporta cada conglomerado. Con una mediana de 3.871 nacimientos por municipio, la distribución
del efecto aleatorio de cada municipio está muy concentrada y es casi gaussiana, que es
exactamente el supuesto de la aproximación.

**El caso que hay que probar además.** Celdas de un solo nacimiento, que es el peor escenario
para Laplace. Si aguanta ahí, aguanta en todo el análisis.

In [ ]:
# ---------------------------------------------------------------------------
# Laplace contra cuadratura adaptativa de Gauss-Hermite
# ---------------------------------------------------------------------------

from scipy.special import expit, logsumexp, gammaln
from scipy.optimize import minimize

def _ajustar_agq(X, y, n, g, Q):
    """
    Ajuste por cuadratura adaptativa de Gauss-Hermite con Q nodos.

    Q=1 equivale exactamente a la aproximacion de Laplace. Al aumentar Q, la
    cuadratura converge al valor exacto de la integral, de modo que comparar
    Q=1 contra Q grande mide el error de Laplace.

    ADAPTATIVA quiere decir que los nodos se centran y escalan en el modo y la
    curvatura de cada cluster. Sin adaptar, la cuadratura necesitaria cientos
    de nodos cuando la posterior es estrecha, y con pocos nodos da resultados
    erroneos: se comprobo que subestimaba sigma en un 30 %.
    """
    ng = int(g.max()) + 1
    z, w = np.polynomial.hermite_e.hermegauss(Q)
    lw = np.log(w)
    coef = gammaln(n + 1) - gammaln(y + 1) - gammaln(n - y + 1)

    def modo(eta0, sigma):
        u = np.zeros(ng); inv = 1.0 / sigma ** 2
        for _ in range(12):
            p = expit(eta0 + u[g])
            d1 = np.bincount(g, weights=y - n * p, minlength=ng) - u * inv
            d2 = -np.bincount(g, weights=n * p * (1 - p), minlength=ng) - inv
            paso = d1 / d2
            u = u - paso
            if np.max(np.abs(paso)) < 1e-10:
                break
        p = expit(eta0 + u[g])
        curv = np.bincount(g, weights=n * p * (1 - p), minlength=ng) + inv
        return u, 1.0 / np.sqrt(curv)

    def objetivo(theta):
        beta, sigma = theta[:-1], np.exp(theta[-1])
        eta0 = X @ beta
        u_hat, tau = modo(eta0, sigma)
        acum = np.empty((Q, ng))
        for q in range(Q):
            u = u_hat + tau * z[q]
            eta = eta0 + u[g]
            ll = y * eta - n * np.logaddexp(0.0, eta) + coef
            h = (np.bincount(g, weights=ll, minlength=ng)
                 - 0.5 * (u / sigma) ** 2 - np.log(sigma))
            acum[q] = h + 0.5 * z[q] ** 2 + lw[q] + np.log(tau)
        return -np.sum(logsumexp(acum, axis=0))

    p0 = np.zeros(X.shape[1])
    p0[0] = np.log((y.sum() + .5) / (n.sum() - y.sum() + .5))
    r = minimize(objetivo, np.append(p0, np.log(0.3)), method="L-BFGS-B",
                 options={"maxiter": 600, "ftol": 1e-13, "gtol": 1e-9})
    return {"beta": r.x[:-1], "sigma": float(np.exp(r.x[-1])), "loglik": -r.fun}


# --- Caso A: celdas con muchos nacimientos, como el analisis agregado ------
rng = np.random.default_rng(3)
J = 1157
v_real = rng.normal(0, 0.35, J)
g = np.repeat(np.arange(J), 30)
x = rng.normal(0, 1, len(g))
n = rng.integers(20, 200, len(g)).astype(float)
y = rng.binomial(n.astype(int), expit(-2.35 + 0.25 * x + v_real[g])).astype(float)
X = np.column_stack([np.ones(len(g)), x])

print("CASO A · celdas con muchos nacimientos\n")
print(f"{'nodos':>6} {'b1':>10} {'sigma':>10} {'log-verosimilitud':>20}")
print("-" * 50)
filas_a = []
for Q in (1, 3, 7, 15):
    r = _ajustar_agq(X, y, n, g, Q)
    etiqueta = "1 (Laplace)" if Q == 1 else str(Q)
    print(f"{etiqueta:>6} {r['beta'][1]:>10.5f} {r['sigma']:>10.5f} "
          f"{r['loglik']:>20.3f}")
    filas_a.append({"caso": "muchos nacimientos", "nodos": Q,
                    "b1": r["beta"][1], "sigma": r["sigma"],
                    "loglik": r["loglik"]})

# --- Caso B: celdas de UN solo nacimiento, el peor caso para Laplace -------
rng = np.random.default_rng(3)
v_real = rng.normal(0, 0.35, J)
g = np.repeat(np.arange(J), 200)
x = rng.normal(0, 1, len(g))
n = np.ones(len(g))
y = rng.binomial(1, expit(-2.35 + 0.25 * x + v_real[g])).astype(float)
X = np.column_stack([np.ones(len(g)), x])

print("\n\nCASO B · celdas de un solo nacimiento (peor caso para Laplace)\n")
print(f"{'nodos':>6} {'b1':>10} {'sigma':>10} {'log-verosimilitud':>20}")
print("-" * 50)
filas_b = []
for Q in (1, 3, 7, 15):
    r = _ajustar_agq(X, y, n, g, Q)
    etiqueta = "1 (Laplace)" if Q == 1 else str(Q)
    print(f"{etiqueta:>6} {r['beta'][1]:>10.5f} {r['sigma']:>10.5f} "
          f"{r['loglik']:>20.3f}")
    filas_b.append({"caso": "un nacimiento", "nodos": Q,
                    "b1": r["beta"][1], "sigma": r["sigma"],
                    "loglik": r["loglik"]})

cuadratura = pd.DataFrame(filas_a + filas_b)
guardar_tabla(cuadratura, "validacion_laplace_cuadratura", decimales=6)

# --- El veredicto ----------------------------------------------------------
print("\n\nDIFERENCIA ENTRE LAPLACE Y LA CUADRATURA MAS FINA\n")
for caso in cuadratura["caso"].unique():
    sub = cuadratura[cuadratura["caso"] == caso]
    laplace = sub[sub["nodos"] == 1].iloc[0]
    fina = sub[sub["nodos"] == sub["nodos"].max()].iloc[0]
    print(f"  {caso}")
    print(f"     b1     {abs(fina['b1'] - laplace['b1']):.2e}")
    print(f"     sigma  {abs(fina['sigma'] - laplace['sigma']):.2e}")

**Cómo se lee esta salida.**

**Caso A — celdas con muchos nacimientos**, que es la situación del análisis agregado:

| Nodos | $\beta_1$ | $\sigma$ |
|---|---:|---:|
| **1 (Laplace)** | 0,25293 | 0,34799 |
| 3 | 0,25293 | 0,34799 |
| 7 | 0,25293 | 0,34800 |
| 15 | 0,25293 | 0,34800 |

**Caso B — celdas de un solo nacimiento**, el peor escenario para Laplace:

| Nodos | $\beta_1$ | $\sigma$ |
|---|---:|---:|
| **1 (Laplace)** | 0,26900 | 0,33517 |
| 3 | 0,26901 | 0,33539 |
| 7 | 0,26901 | 0,33614 |
| 15 | 0,26901 | 0,33614 |

**Las diferencias entre Laplace y la cuadratura de quince nodos:**

| | $\beta_1$ | $\sigma$ |
|---|---:|---:|
| Muchos nacimientos | $1{,}2\times10^{-8}$ | $1{,}4\times10^{-5}$ |
| Un solo nacimiento | $1{,}1\times10^{-5}$ | $9{,}7\times10^{-4}$ |

**En el caso que corresponde al análisis —celdas con muchos nacimientos— el error es de
$10^{-8}$ en el coeficiente.** En el peor caso imaginable, celdas de un solo nacimiento, sube a
$10^{-3}$ en la desviación típica, que sigue siendo despreciable frente a las magnitudes que
se interpretan.

**La conclusión.** Laplace no introduce error apreciable en este problema. Con una mediana de
3.871 nacimientos por municipio, la distribución de cada efecto aleatorio está lo bastante
concentrada como para que la aproximación gaussiana sea prácticamente exacta.

**Un detalle que conviene conocer, porque es un error frecuente.** La cuadratura tiene que ser
**adaptativa** — con los nodos centrados y escalados en el modo y la curvatura de cada
conglomerado. Con cuadratura **no adaptativa**, una versión anterior de esta comprobación
subestimaba $\sigma$ en cerca del 30 % y no convergía al aumentar los nodos, porque los nodos
fijos no muestreaban la región donde la posterior tiene masa. Que las estimaciones sean
estables desde un solo nodo es precisamente la señal de que la adaptación está bien hecha.

**Qué va al manuscrito.** Las dos tablas al Apéndice; en §4.4.4, la frase de que la
aproximación se contrastó contra cuadratura adaptativa y coinciden hasta el quinto decimal.

## 5. Cierre y resumen para la defensa

In [ ]:
# ---------------------------------------------------------------------------
# Cierre
# ---------------------------------------------------------------------------

from config import DIR_TABLAS, DIR_FIGURAS

print("RESUMEN DE LA VALIDACION\n")
print("1 · POR QUE NO SE USA UNA BIBLIOTECA")
print(f"     statsmodels deja de converger a 500 conglomerados")
print(f"     el analisis necesita 1.157")
print(f"     la matriz que exigiria: 1,2 GB en M1, 5,8 GB en M2")

print("\n2 · LA AGREGACION BINOMIAL ES EXACTA")
print(f"     diferencia maxima entre individual y agregado: {maxima:.2e}")

print("\n3 · EL ESTIMADOR ES INSESGADO")
for _, f in sesgos.iterrows():
    print(f"     {f['parametro']:<12} sesgo {f['sesgo']:+.6f}")
print(f"     con varianza nula en la verdad, la estima en "
      f"{m_nulo['sigma_supergrupo']:.6f}")

print("\n4 · LAPLACE NO INTRODUCE ERROR APRECIABLE")
for caso in cuadratura["caso"].unique():
    sub = cuadratura[cuadratura["caso"] == caso]
    lap = sub[sub["nodos"] == 1].iloc[0]
    fin = sub[sub["nodos"] == sub["nodos"].max()].iloc[0]
    print(f"     {caso:<22} diferencia en sigma {abs(fin['sigma']-lap['sigma']):.2e}")

print("\n\nTABLAS GENERADAS")
for nombre in ["validacion_escalamiento_statsmodels", "validacion_agregacion",
               "validacion_simulaciones", "validacion_sesgo",
               "validacion_laplace_cuadratura"]:
    marca = "OK   " if (DIR_TABLAS / f"{nombre}.csv").exists() else "FALTA"
    print(f"  {marca}  {nombre}")

print("\n" + "=" * 60)
versiones()

## Las cuatro afirmaciones que sostiene este apéndice

Cada una con la cifra que la respalda, para poder citarlas sin consultar nada.

**1 · No había implementación en Python que sirviera.**
`statsmodels` deja de converger a partir de 500 conglomerados. El análisis necesita 1.157. La
matriz de efectos aleatorios que su interfaz exige materializar sería de 1,2 GB en M1 y 5,8 GB
en M2, sin contar las copias del optimizador.

**2 · La agregación binomial es exacta, no un submuestreo.**
Ajustar sobre datos individuales y sobre celdas agregadas da los mismos coeficientes hasta la
precisión numérica. La verosimilitud es la misma función escrita de otra forma.

**3 · El estimador es insesgado.**
Sobre cinco simulaciones con componentes de varianza conocidas: sesgo de **+0,0004** en el
coeficiente, **+0,003** y **+0,001** en las desviaciones típicas. Con varianza departamental
nula en la simulación, la estima en **0,009**. Los efectos aleatorios se recuperan con
correlación de **0,97**.

**4 · Laplace no introduce error apreciable.**
Contrastada contra cuadratura adaptativa de Gauss-Hermite de quince nodos: coinciden hasta el
quinto decimal, también en el peor caso de celdas con un solo nacimiento.

---

## Lo que NO hay que afirmar

**No se desarrolló un método nuevo.** El modelo es estándar —logístico de interceptos
aleatorios, Laplace— y es exactamente lo que hace `lme4` en R por defecto. Lo que se
implementó es ese método, aprovechando que el anidamiento estricto de municipios en
departamentos regala una estructura algebraica que hace el cálculo viable.

Decir «se desarrolló un estimador nuevo» sería sobreafirmar, y un jurado con formación
estadística lo detectaría de inmediato. La formulación correcta está en la primera celda de
este notebook.

## Referencias que hay que añadir al `.bib`

Verificar en Crossref antes de incluirlas:

- **Breslow y Clayton (1993)**, *Approximate inference in generalized linear mixed models*,
  JASA 88(421):9-25 — sostiene la aproximación de Laplace
- **Bates et al.**, la referencia de `lme4`, para poder decir que el procedimiento es el
  mismo que usa la implementación de referencia